# 🧪 Ensemble Learning Lab: Bagging, Boosting, and Stacking (Classification)

## 🎯 Objective
In this task, you will **design, train, evaluate, and compare ensemble learning models** for a **classification problem** using real-world data.  
You will work with **Bagging**, **Boosting**, and **Stacking** techniques and analyze their performance using appropriate evaluation metrics.

---

## 📊 Dataset
- **Source**: :contentReference[oaicite:0]{index=0}  
- **Competition**: *Playground Series – Season 6, Episode 2*  
- **Problem Type**: **Supervised Classification**

🔗 Dataset link:  
[URL](https://www.kaggle.com/competitions/playground-series-s6e2/overview)

> **Note**: You must download the dataset using the Kaggle API or manually upload it to Colab/Jupyter.

---

## 🧠 Models to Implement

### 1️⃣ Bagging
- **Model**: `BaggingClassifier`
- **Base Estimator**: Decision Tree (recommended)
- Key hyperparameters to explore:
  - `n_estimators`
  - `max_samples`
  - `max_features`
  - `bootstrap`

---

### 2️⃣ Boosting Models

#### a) AdaBoost
- **Model**: `AdaBoostClassifier`
- Base estimator: Decision Tree (stump recommended)
- Tune:
  - `n_estimators`
  - `learning_rate`

#### b) Gradient Boosting
- **Model**: `GradientBoostingClassifier`
- Tune:
  - `n_estimators`
  - `learning_rate`
  - `max_depth`
  - `subsample`

#### c) XGBoost
- **Model**: `XGBClassifier`
- Tune:
  - `n_estimators`
  - `learning_rate`
  - `max_depth`
  - `subsample`
  - `colsample_bytree`

> ⚠️ Handle class imbalance if present.

---

### 3️⃣ Stacking
- **Model**: `StackingClassifier`
- **Base learners** (example):
  - Logistic Regression
  - Random Forest
  - Gradient Boosting
- **Meta-learner**:
  - Logistic Regression (recommended)

---

## ⚙️ Task Instructions

### 🔹 Step 1: Data Preparation
- Load training data
- Separate features and target
- Handle missing values (if any)
- Encode categorical variables
- Perform train/validation split
- Apply feature scaling where necessary

---

### 🔹 Step 2: Model Training
- Train **each ensemble model independently**
- Use **cross-validation** where appropriate
- Record training time and key hyperparameters

---

### 🔹 Step 3: Evaluation Metrics
Evaluate all models using:
- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**
- **ROC-AUC**
- **Confusion Matrix**

---

### 🔹 Step 4: Model Comparison
Create a **comparison table** that includes:
- Model name
- Accuracy
- F1-score
- ROC-AUC
- Training time
- Key observations

---

## 📈 Analysis & Discussion (Required)

Answer the following:
1. Which ensemble method performed best and why?
2. How does **bagging** differ from **boosting** in terms of bias and variance?
3. Did stacking outperform individual ensemble models?
4. Which model would you choose for deployment and why?
5. What are the computational trade-offs between these methods?

---

## 📝 Deliverables
- Fully executable notebook
- Clean, well-documented code
- Final comparison table
- Written analysis and conclusions

---

## ⭐ Bonus (Optional)
- Perform **hyperparameter tuning** using GridSearchCV or RandomizedSearchCV
- Plot **ROC curves** for all models
- Analyze **feature importance** for boosting models

---

🎓 **Learning Outcome**  
By completing this task, you will gain hands-on experience with advanced ensemble techniques and develop a strong intuition for **when and why to use bagging, boosting, or stacking in classification problems**.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from google.colab import files
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import time

In [2]:
uploaded = files.upload()

df = pd.read_csv("train.csv")

Saving train.csv to train.csv


In [3]:
df.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Age                      630000 non-null  int64  
 2   Sex                      630000 non-null  int64  
 3   Chest pain type          630000 non-null  int64  
 4   BP                       630000 non-null  int64  
 5   Cholesterol              630000 non-null  int64  
 6   FBS over 120             630000 non-null  int64  
 7   EKG results              630000 non-null  int64  
 8   Max HR                   630000 non-null  int64  
 9   Exercise angina          630000 non-null  int64  
 10  ST depression            630000 non-null  float64
 11  Slope of ST              630000 non-null  int64  
 12  Number of vessels fluro  630000 non-null  int64  
 13  Thallium                 630000 non-null  int64  
 14  Hear

In [5]:
y = df["Heart Disease"].map({"Presence":1, "Absence":0})

In [6]:
X = df.drop(["id","Heart Disease"], axis=1)

In [7]:
for col in X.select_dtypes(include="object").columns:
  le = LabelEncoder()
  X[col] = le.fit_transform(X[col])

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [10]:
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(),n_estimators=100,max_samples=0.8,max_features=0.8,bootstrap=True,random_state=42)

In [11]:
adaboost = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),n_estimators=200,learning_rate=0.05,random_state=42)

In [12]:
gb = GradientBoostingClassifier(n_estimators=200,learning_rate=0.05,max_depth=3,subsample=0.8,random_state=42)

In [13]:
xgb = XGBClassifier(n_estimators=300,learning_rate=0.05,max_depth=4,subsample=0.8,colsample_bytree=0.8,random_state=42,use_label_encoder=False,eval_metric='logloss')

In [14]:
stacking = StackingClassifier(
    estimators=[('lr', LogisticRegression(max_iter=1000)),('rf', RandomForestClassifier(n_estimators=100)),('gb', GradientBoostingClassifier(n_estimators=100))],final_estimator=LogisticRegression(max_iter=1000))

In [15]:
def evaluate_model(model, X_train, X_val, y_train, y_val):
    start = time.time()
    model.fit(X_train, y_train)
    end = time.time()

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:,1] if hasattr(model, "predict_proba") else None

    results = {
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1-score": f1_score(y_val, y_pred),
        "ROC-AUC": roc_auc_score(y_val, y_prob) if y_prob is not None else None,
        "Training Time": round(end - start, 2),
        "Confusion Matrix": confusion_matrix(y_val, y_pred)
    }
    return results


In [16]:
models = {"Bagging": bagging,"AdaBoost": adaboost,"GradientBoosting": gb,"XGBoost": xgb,"Stacking": stacking}

In [17]:
results = {}
for name, model in models.items():
    results[name] = evaluate_model(model, X_train, X_val, y_train, y_val)

comparison = pd.DataFrame(results).T.drop("Confusion Matrix", axis=1)
print(comparison)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:44:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


                  Accuracy Precision    Recall  F1-score   ROC-AUC  \
Bagging           0.885048  0.882028   0.85842  0.870064  0.951286   
AdaBoost          0.879516  0.893624  0.830079   0.86068  0.949406   
GradientBoosting  0.888516  0.882875  0.866262   0.87449  0.954558   
XGBoost           0.889024  0.882762  0.867713  0.875173  0.955574   
Stacking          0.888405  0.882652  0.866262   0.87438  0.954535   

                 Training Time  
Bagging                 169.98  
AdaBoost                 66.06  
GradientBoosting        143.36  
XGBoost                  17.45  
Stacking                905.48  


## **1-Stacking achieved the highest accuracy and ROC-AUC, slightly outperforming XGBoost. This is expected since stacking leverages multiple models’ strengths.**

**2-Bagging reduces variance by averaging predictions from multiple models trained independently.**

**Boosting reduces bias by sequentially correcting errors of weak learners.**


**3-Yes, stacking outperformed Bagging, AdaBoost, Gradient Boosting, and even XGBoost, showing the benefit of combining diverse learners**

**4-XGBoost is often chosen in practice because it balances speed, accuracy, and interpretability (feature importance). Stacking is powerful but more complex to maintain**

**5- Bagging: Fast, parallelizable, but less accurate.**

**Boosting: More accurate, but slower and risk of overfitting.**

**Stacking: Highest accuracy, but computationally expensive and harder to deploy.**